In [1]:
# Torch version
!python -c "import torch; print(torch.__version__)"

# Cuda version
!python -c "import torch; print(torch.version.cuda)"

2.6.0+cu124
12.4


In [2]:
# Uninstall
# !pip uninstall torch-scatter torch-sparse torch-cluster torch-spline-conv pyg-lib -y

In [3]:
# Update Torch
# !pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

In [4]:
# Install PyG (automatic)
# !pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{torch.__version__}.html
# !pip install torch_geometric

In [5]:
# Verify instalation
import torch
import torch_geometric
import torch_scatter

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch_scatter.__version__)
print(torch_geometric.__version__)


2.6.0+cu124
12.4
True
2.1.2+pt26cu124
2.7.0


In [6]:
from model_PyG_cla import *
from utils import *

In [7]:
import json
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import time
import torch
import torch.nn.functional as F
import torch_geometric.transforms as T

from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import InMemoryDataset, Data
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import Compose
from torch_geometric.utils import dense_to_sparse, negative_sampling
from torch.nn.functional import binary_cross_entropy_with_logits
from torch.optim import Adam

In [8]:
import torch_geometric
print(torch_geometric.__version__)

np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

2.7.0


### Parameters

In [9]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"] # Change to static, e.g. "exp1"

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

raw_data_file = params["raw_data_file"]
print("Raw data:\t", raw_data_file)

methods = params["methods"]
print("Methods:\t", methods)

apply_transformation = params["apply_transformation"]
print("Has transformation:", apply_transformation)

dimension = params["dimension"]
print("Dimension:\t", dimension)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

cuda = params["cuda"]
print("Cuda:\t", cuda)

epochs = params["epochs"]
print("Epochs:\t", epochs)

lr = params["lr"]
print("Lr:\t", lr)

weight_decay = params["weight_decay"]
print("Weight decay:\t", weight_decay)

patience = params["patience"]
print("Patience:\t", patience)

Exp:		 exp108
Raw data:	 douban
Methods:	 ['t-gae']
Has transformation: False
Dimension:	 512
Groups id:	 ['douban']
Subgroups id:	 {'douban': ['1', '2']}
Cuda:	 1
Epochs:	 500
Lr:	 0.0001
Weight decay:	 0.0001
Patience:	 10


### Setup

In [10]:
# Parameters models

dataset = exp
encoders = ["GCNRes", "GIN", "GINE", "GCN", "GAT", "GATE"] # ["GIN", "GINE"] # ["GIN", "GINE"] # ["GIN", "GINE"] # Change
device = torch.device(f"cuda:{cuda}" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

NUM_HIDDEN_LAYERS = 12
HIDDEN_DIM = [1024] * NUM_HIDDEN_LAYERS + [1024]
output_feature_size = dimension # 128
# lr = 0.001
# epochs = 100

In [11]:
# Create datasets (groups and subgroups)

list_train_set = []

for group_id in groups_id:
	train_set = []
	for subgroup_id in subgroups_id[group_id]:
		train_set.append("{}_{}".format(group_id, subgroup_id))
	list_train_set.append(train_set)
list_train_set

[['douban_1', 'douban_2']]

### Create Data (PyG)

In [12]:
# Only for GIN
""" transform = Compose([
	# T.NormalizeFeatures(),
	T.ToUndirected(reduce="mean"),
	T.AddSelfLoops(fill_value=1.0),
	T.ToDevice(device)
]) """

# For GIN and GINE
transform = T.Compose([
	# T.NormalizeFeatures(),
	T.ToUndirected(reduce="mean"),
	# T.AddSelfLoops(attr="edge_attr", fill_value="mean"),
	# T.AddLaplacianEigenvectorPE(k=3, attr_name=None, is_undirected=True),
	# T.AddRandomWalkPE(walk_length=8, attr_name=None),
	T.ToDevice(device)
])

In [13]:
print("Loading training datasets")
has_edge_attributes = False
with_mean1 = True
with_mean2 = True

list_train_loader = []

if raw_data_file == "ACM_DBLP":
	group_id = groups_id[0]
	
	# Read dataset
	b = np.load("data/ACM-DBLP.npz")

	# Node matching
	test_pairs = b["test_pairs"].astype(np.int32)
	pos_pairs = b["pos_pairs"].astype(np.int32)
	df_node_alignment_truth = pd.DataFrame(np.concatenate((test_pairs, pos_pairs), axis=0))
	df_node_alignment_truth

	# Data preparation
	subgroup_id = 1
	edge_index1 = torch.tensor(b[f"edge_index{subgroup_id}"], dtype=torch.long)
	x1 = torch.tensor(b[f"x{subgroup_id}"], dtype=torch.float)

	subgroup_id = 2
	edge_index2 = torch.tensor(b[f"edge_index{subgroup_id}"], dtype=torch.long)
	x2 = torch.tensor(b[f"x{subgroup_id}"], dtype=torch.float)

elif raw_data_file == "Douban Online_Offline":
	group_id = groups_id[0]
	
	# Read dataset
	a1, f1, a2, f2, test_pairs = load_douban()

	# Node matching
	test_pairs = torch.tensor(np.array(test_pairs, dtype=int)) - 1
	test_pairs = test_pairs.numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs.T)
	df_node_alignment_truth

	# Create data
	edge_index1, _ = dense_to_sparse(torch.from_numpy(a1.toarray()))
	x1 = torch.from_numpy(f1.toarray()).float()

	edge_index2, _= dense_to_sparse(torch.from_numpy(a2.toarray()))
	x2 = torch.from_numpy(f2.toarray()).float()

	with_mean1 = False
	with_mean2 = False

elif raw_data_file == "Cora1-Cora2":
	group_id = groups_id[0]
		
	# Read dataset
	loader = load_npz("data/cora.npz")
	data = loader["adj_matrix"]
	samples = data.shape[0]
	features = data.shape[1]
	values = data.data
	coo_data = data.tocoo()
	# indices = torch.LongTensor([coo_data.row, coo_data.col])
	indices = np.array([coo_data.row, coo_data.col])

	# Node matching
	N = 2708
	test_pairs = [[i, i] for i in range(N)]
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	node_features = loader["node_attr"]
	node_features = node_features.toarray()

	edge_index1 = torch.tensor(indices, dtype=torch.long)
	x1 = torch.tensor(node_features, dtype=torch.float)

	edge_index2 = torch.tensor(indices, dtype=torch.long)
	x2 = torch.tensor(node_features, dtype=torch.float)

	with_mean1 = False
	with_mean2 = True

elif raw_data_file == "flickr-lastfm":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/flickr-lastfm.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = True
	with_mean1 = False
	with_mean2 = False

elif raw_data_file == "douban":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/douban.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = True
	with_mean1 = False
	with_mean2 = False

elif raw_data_file == "flickr-myspace":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/flickr-myspace.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = True
	with_mean1 = False
	with_mean2 = False

elif raw_data_file == "ACM-DBLP":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/ACM-DBLP.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	# edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	# edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	# edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = False
	with_mean1 = True
	with_mean2 = True

elif raw_data_file == "cora":
	# https://github.com/yq-leo/PlanetAlign/blob/main/PlanetAlign/datasets/flickr_lastfm.py
	data = torch.load("data/cora.pt")
	edges = data["edges"]
	node_features = data["node_attributes"]
	# edges_attributes = data["edge_attributes"]
	
	# Node matching
	test_pairs = data["anchor_links"].cpu().numpy()
	df_node_alignment_truth = pd.DataFrame(test_pairs)
	df_node_alignment_truth

	# Create data
	edge_index1 = edges[0]
	x1 = torch.tensor(node_features[0].cpu().numpy(), dtype=torch.float)
	# edge_attr1 = torch.tensor(edges_attributes[0].cpu().numpy(), dtype=torch.float)

	edge_index2 = edges[1]
	x2 = torch.tensor(node_features[1].cpu().numpy(), dtype=torch.float)
	# edge_attr2 = torch.tensor(edges_attributes[1].cpu().numpy(), dtype=torch.float)

	has_edge_attributes = False
	with_mean1 = False
	with_mean2 = True
	
# Create data
train_loader = {}

# Include edge_attr
subgroup_id = 1
data1 = Data(x=x1, edge_index=edge_index1)
if has_edge_attributes:
	data1.edge_attr = edge_attr1
else:
	data1 = add_edge_attributes(data1)
print(subgroup_id, data1)
info_data(data1, include_vectors=True)

subgroup_id = 2
data2 = Data(x=x2, edge_index=edge_index2)
if has_edge_attributes:
	data2.edge_attr = edge_attr2
else:
	data2 = add_edge_attributes(data2)
print(subgroup_id, data2)
info_data(data2, include_vectors=True)

# Normalize
normalize_node_edge_features(data1, data2, with_mean1=with_mean1, with_mean2=with_mean2) # Important

# Apply transform
subgroup_id = 1
data1 = transform(data1)
print(subgroup_id, data1)
# info(data1)

subgroup_id = 2
data2 = transform(data2)
print(subgroup_id, data2)
# info(data2)

# [MEJORA 3] Antes: gcn_norm(..., add_self_loops=False) -> edge_weight = D^-1/2 A D^-1/2 (SIN la identidad).
# El script original usa A_hat = D^-1/2 (A+I) D^-1/2 (preprocess_graph). Ahora el modelo "ORI"/GCNRes calcula
# A_hat internamente con self-loops (gcn_norm(add_self_loops=True)); GIN/GINE no usan edge_weight.
# data.edge_index queda intacto (mismas aristas y mismo orden que edge_attr).
subgroup_id = 1
train_loader[f"{group_id}_{subgroup_id}"] = data1
print(1, data1)

subgroup_id = 2
train_loader[f"{group_id}_{subgroup_id}"] = data2
print(2, data2)

info_data(data1, include_vectors=True)
info_data(data2, include_vectors=True)

list_train_loader.append(train_loader)

Loading training datasets
1 Data(x=[3906, 538], edge_index=[2, 16328], edge_attr=[16328, 2])
Validate:	 True
Num. nodes:	 3906
Num. edges:	 16328
Num. features:	 538
Has isolated:	 False
Has loops:	 False
Is directed:	 True
Is undirected:	 False
tensor([[   0,    0,    1,  ..., 3904, 3905, 3905],
        [ 728, 3714,  160,  ..., 3671, 2166, 3790]])
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])
tensor([[0., 1.],
        [0., 1.],
        [0., 1.],
        ...,
        [1., 0.],
        [0., 1.],
        [0., 1.]])
2 Data(x=[1118, 538], edge_index=[2, 3022], edge_attr=[3022, 2])
Validate:	 True
Num. nodes:	 1118
Num. edges:	 3022
Num. features:	 538
Has isolated:	 False
Has loops:	 False
Is directed:	 False
Is undirected:	 True
tensor([[   0,    1,    1,  ..., 1117, 1117, 1117],
    

In [14]:
df_node_alignment_truth

,0,1
0,831,322
1,1033,537
2,3328,453
3,2535,795
4,3115,612
...,...,...
1113,3309,473
1114,1051,325
1115,2819,396
1116,2869,757


### Train

In [15]:
"""Entrenamiento T-GAE para subgraph alignment (PyTorch Geometric).

Requiere el modelo refactorizado de model_PyG.py (edge_weight / edge_attr por nombre).

Cambios respecto a la versión anterior de fit_TGAE_subgraph:
  1. Los positivos de la reconstrucción son las aristas ORIGINALES sin self-loops
	 (data.pos_edge_index), no data.edge_index (que trae self-loops de AddSelfLoops).
  2. Selección de modelo opcional por alineamiento en un conjunto de validación
	 (val_pairs) y restauración de los mejores pesos. Sin val_pairs: early stopping
	 por pérdida, como antes.
  3. Decoder opcional "cosine" (coseno / tau + sesgo aprendible) además del producto interno.
  4. Aumentación opcional por eliminación de aristas (aug_p > 0): se codifica el grafo
	 perturbado y se reconstruye el grafo original (Ec. 7 del paper, aplicada por grafo).
  5. Pérdida balanceada: 0.5 * (BCE positivos + BCE negativos), con neg_ratio configurable.
  6. Opcional: recorte de gradiente, AdamW, semilla.
  7. Métricas de alineamiento (Hit@K, MRR) con similitud coseno, L2 o CSLS.
"""
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.utils import negative_sampling, remove_self_loops
from tqdm import tqdm


GCNRES_NAMES = ("GCNRes", "ORI")  # "ORI" es el nombre que usa el notebook para la capa original

# ----------------------------------------------------------------------------
# Utilidades
# ----------------------------------------------------------------------------
def set_seed(seed):
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)


def make_val_split(pairs, val_frac=0.2, seed=0):
	"""Divide pares de anclaje [K, 2] en (validación, evaluación).

	Reporta las métricas finales SOLO sobre el segundo conjunto.
	"""
	pairs = np.asarray(pairs)
	idx = np.random.default_rng(seed).permutation(len(pairs))
	n_val = int(round(val_frac * len(pairs)))
	return pairs[idx[:n_val]], pairs[idx[n_val:]]


def prepare_graph(data):
	"""Guarda las aristas originales sin self-loops (objetivo de reconstrucción)."""
	if getattr(data, "pos_edge_index", None) is None:
		data.pos_edge_index, _ = remove_self_loops(data.edge_index)
	return data


def drop_undirected_edges(edge_index, edge_attr, p, num_nodes):
	"""Elimina cada arista no dirigida con probabilidad p (ambos sentidos a la vez).

	Conserva los self-loops y mantiene edge_attr alineado con edge_index.
	"""
	row, col = edge_index
	key = torch.minimum(row, col) * num_nodes + torch.maximum(row, col)
	uniq, inv = torch.unique(key, return_inverse=True)
	keep_undirected = torch.rand(uniq.numel(), device=edge_index.device) >= p
	keep = keep_undirected[inv] | (row == col)
	edge_index = edge_index[:, keep]
	if edge_attr is not None:
		edge_attr = edge_attr[keep]
	return edge_index, edge_attr


def _encode(model, data, encoder, aug_p=0.0):
	"""Llama al modelo según el encoder. GCNRes recibe aristas sin loops y el modelo
	calcula Â = D^-1/2 (A+I) D^-1/2 internamente; GCN recibe pesos crudos (ninguno)."""
	ei = data.pos_edge_index if encoder in GCNRES_NAMES else data.edge_index
	ea = data.edge_attr if encoder in ("GINE", "GATE") else None
	if aug_p > 0:
		ei, ea = drop_undirected_edges(ei, ea, aug_p, data.num_nodes)
	kw = {"edge_attr": ea} if ea is not None else {}
	return model(data.x, ei, **kw)


def _edge_logits(z, pos, neg, decoder, tau, bias):
	if decoder == "cosine":
		z = F.normalize(z, dim=1)
	pos_logits = (z[pos[0]] * z[pos[1]]).sum(dim=1)
	neg_logits = (z[neg[0]] * z[neg[1]]).sum(dim=1)
	if decoder == "cosine":
		pos_logits = pos_logits / tau + bias
		neg_logits = neg_logits / tau + bias
	return pos_logits, neg_logits


# ----------------------------------------------------------------------------
# Evaluación de alineamiento
# ----------------------------------------------------------------------------
def _mean_topk(a, b, k, chunk=2048):
	"""Para cada fila de a: media de las k mayores similitudes coseno con las filas de b."""
	out = []
	for i in range(0, a.size(0), chunk):
		out.append((a[i:i + chunk] @ b.T).topk(k, dim=1).values.mean(dim=1))
	return torch.cat(out)


@torch.no_grad()
def similarity_matrix(z1, z2, retrieval="cosine", csls_k=None):
    """Full similarity matrix [N1, N2] (higher = more similar), as PlanetAlign expects."""
    if retrieval == "cosine":
        a, b = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
        sim = a @ b.T
        if csls_k:
            r_s = _mean_topk(a, b, csls_k)
            r_t = _mean_topk(b, a, csls_k)
            sim = 2 * sim - r_s.unsqueeze(1) - r_t.unsqueeze(0)
        return sim
    if retrieval == "l2":
        if csls_k:
            raise ValueError("csls_k is only supported with retrieval='cosine'")
        return -torch.cdist(z1, z2)  # distances -> similarities
    raise ValueError("retrieval must be 'cosine' or 'l2'")

@torch.no_grad()
def evaluate_alignment(z1, z2, pairs, retrieval="cosine", csls_k=None, ks=(1, 5, 10, 50), mode="ltr"):
    """Hit@K and MRR computed with PlanetAlign's hits_ks_scores and mrr_score.

    pairs: [K, 2] (idx in graph 1, idx in graph 2).
    mode: 'ltr' (graph1->graph2), 'rtl', 'mean' or 'max', as in PlanetAlign.
    Returns {"hit@k": ..., "mrr": ...}.
    """
    sim = similarity_matrix(z1, z2, retrieval, csls_k)
    pairs = torch.as_tensor(pairs, dtype=torch.long, device=sim.device)
    hits = hits_ks_scores(sim, pairs, ks=ks, mode=mode)
    out = {f"hit@{k}": hits[k] for k in ks}
    out["mrr"] = mrr_score(sim, pairs, mode=mode)
    return out
 
 
def evaluate_alignment_mean(z1, z2, pairs, **kw):
    """evaluate_alignment with mode="mean" (average of both directions, as in the notebook)."""
    return evaluate_alignment(z1, z2, pairs, mode="mean", **kw)


@torch.no_grad()
def evaluate_alignment_v1(z1, z2, pairs, retrieval="cosine", csls_k=None, ks=(1, 5, 10, 50)):
	"""Hit@K y MRR de z1 (grafo 1) hacia z2 (grafo 2). pairs: [K, 2] (idx en 1, idx en 2)."""
	pairs = torch.as_tensor(np.asarray(pairs), dtype=torch.long, device=z1.device)
	src, tgt = pairs[:, 0], pairs[:, 1]
	if retrieval == "cosine":
		a, b = F.normalize(z1, dim=1), F.normalize(z2, dim=1)
		sim = a[src] @ b.T
		if csls_k:
			r_s = _mean_topk(a[src], b, csls_k)
			r_t = _mean_topk(b, a, csls_k)
			sim = 2 * sim - r_s.unsqueeze(1) - r_t.unsqueeze(0)
	elif retrieval == "l2":
		if csls_k:
			raise ValueError("csls_k solo está soportado con retrieval='cosine'")
		sim = -torch.cdist(z1[src], z2)
	else:
		raise ValueError("retrieval debe ser 'cosine' o 'l2'")
	true = sim[torch.arange(len(src), device=sim.device), tgt].unsqueeze(1)
	rank = (sim > true).sum(dim=1) + 1
	out = {f"hit@{k}": (rank <= k).float().mean().item() for k in ks}
	out["mrr"] = (1.0 / rank.float()).mean().item()
	print(1, out)
	# return out

	print(type(sim))
	print(sim)

	# Based on PlanetAlign
	hits = hits_ks_scores(sim.clone().detach(), torch.tensor(EVAL_PAIRS), ks=ks,  mode="mean")
	print(10)
	mrr = mrr_score(sim.clone().detach(), torch.tensor(EVAL_PAIRS), mode="mean")
	print(11)
	hits["mrr"] = mrr
	print(2, hits)
	return hits


def evaluate_alignment_mean_v1(z1, z2, pairs, **kw):
	"""Promedio de grafo1->grafo2 y grafo2->grafo1 (equivale a mode="mean" de PlanetAlign)."""
	pairs = np.asarray(pairs)
	a = evaluate_alignment_v1(z1, z2, pairs, **kw)
	b = evaluate_alignment_v1(z2, z1, pairs[:, ::-1].copy(), **kw)
	return {k: (a[k] + b[k]) / 2 for k in a}


# ----------------------------------------------------------------------------
# Entrenamiento
# ----------------------------------------------------------------------------
def fit_TGAE_subgraph(model, encoder, exp, group_id, subgroups_id, train_loader, train_set,
					no_samples, epochs, lr, weight_decay,
					val_pairs=None, eval_every=5, select_metric="mrr",
					retrieval="cosine", csls_k=None,
					decoder="dot", tau=0.2, neg_ratio=1.0,
					aug_p=0.0, grad_clip=None, optimizer_name="adam",
					patience=50, delta=1e-4, warmup=10, seed=None, save_outputs=True):
	"""Entrena el encoder y guarda embeddings y pérdida (mismas rutas que antes).

	no_samples se conserva por compatibilidad de firma; la pérdida se promedia
	sobre len(train_set). Devuelve un dict con el resumen del entrenamiento.
	"""
	if seed is not None:
		set_seed(seed)
	dev = next(model.parameters()).device
	if val_pairs is not None and len(subgroups_id) < 2:
		raise ValueError("La validación requiere dos subgrafos en subgroups_id")

	for ts in train_set:
		prepare_graph(train_loader[ts])
	for sg in subgroups_id:
		prepare_graph(train_loader[f"{group_id}_{sg}"])

	# Sesgo aprendible solo para el decoder coseno (sin él, la pérdida satura).
	bias = torch.zeros((), device=dev, requires_grad=True) if decoder == "cosine" else None
	param_groups = [{"params": list(model.parameters())}]
	if bias is not None:
		param_groups.append({"params": [bias], "weight_decay": 0.0})
	opt_cls = torch.optim.AdamW if optimizer_name == "adamw" else torch.optim.Adam
	optimizer = opt_cls(param_groups, lr=lr, weight_decay=weight_decay)

	use_val = val_pairs is not None
	list_loss, history = [], []
	best_score, best_epoch, best_state = -float("inf"), 0, None
	best_loss, bad_count, stopped_epoch = None, 0, None
	max_bad = max(1, patience // eval_every) if use_val else patience

	model.train()
	loop_obj = tqdm(range(1, epochs + 1))
	for epoch in loop_obj:
		loop_obj.set_description(f"Epoch: {epoch}")
		optimizer.zero_grad()
		loss = torch.zeros((), device=dev)

		for ts in train_set:
			data = train_loader[ts]
			z = _encode(model, data, encoder, aug_p)

			pos = data.pos_edge_index  # objetivo: grafo original, sin self-loops
			neg = negative_sampling(
				edge_index=pos, num_nodes=z.size(0),
				num_neg_samples=int(neg_ratio * pos.size(1)), method="sparse",
			)
			pos_logits, neg_logits = _edge_logits(z, pos, neg, decoder, tau, bias)
			loss = loss + 0.5 * (
				F.binary_cross_entropy_with_logits(pos_logits, torch.ones_like(pos_logits))
				+ F.binary_cross_entropy_with_logits(neg_logits, torch.zeros_like(neg_logits))
			)

		loss = loss / len(train_set)
		loss.backward()
		if grad_clip is not None:
			torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
		optimizer.step()

		loss_value = loss.item()
		list_loss.append(loss_value)
		loop_obj.set_postfix_str(f"Loss: {loss_value:.4f}")

		if use_val:
			if epoch % eval_every == 0 or epoch == epochs:
				model.eval()
				with torch.no_grad():
					z1 = _encode(model, train_loader[f"{group_id}_{subgroups_id[0]}"], encoder)
					z2 = _encode(model, train_loader[f"{group_id}_{subgroups_id[1]}"], encoder)
					metrics = evaluate_alignment(z1, z2, val_pairs, retrieval, csls_k)
				model.train()
				metrics["epoch"] = epoch
				history.append(metrics)
				score = metrics[select_metric]
				loop_obj.set_postfix_str(f"Loss: {loss_value:.4f} val_{select_metric}: {score:.4f}")
				if score > best_score + 1e-6:
					best_score, best_epoch, bad_count = score, epoch, 0
					# En CPU para no duplicar memoria de GPU.
					best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
				else:
					bad_count += 1
				if epoch >= warmup and bad_count >= max_bad:
					stopped_epoch = epoch
					print(f"Early stopping (validación) en la época {epoch}; mejor época: {best_epoch}")
					break
		else:
			if epoch >= warmup:
				if best_loss is None or loss_value < best_loss - delta:
					best_loss, bad_count = loss_value, 0
				else:
					bad_count += 1
					if bad_count >= max_bad:
						stopped_epoch = epoch
						print(f"Early stopping (pérdida) en la época {epoch}")
						break

	if use_val and best_state is not None:
		model.load_state_dict(best_state)
		print(f"Pesos restaurados de la época {best_epoch} (val_{select_metric}={best_score:.4f})")

	# Embeddings finales
	model.eval()
	list_df_node_embeddings = []
	with torch.no_grad():
		for subgroup_id in subgroups_id:
			data = train_loader[f"{group_id}_{subgroup_id}"]
			z = _encode(model, data, encoder)
			df = pd.DataFrame(z.cpu().numpy())
			df["subgroup_id"] = [subgroup_id] * len(df)
			df.insert(0, "Id", range(len(df)))  # solo para datos que no son MS
			list_df_node_embeddings.append(df)
	df_node_embeddings_concat = pd.concat(list_df_node_embeddings, ignore_index=True)

	if save_outputs:
		emb_dir = f"experiments/output/{exp}/node_embeddings"
		loss_dir = f"experiments/output/{exp}/loss"
		os.makedirs(emb_dir, exist_ok=True)
		os.makedirs(loss_dir, exist_ok=True)
		df_node_embeddings_concat.to_csv(f"{emb_dir}/{encoder}_{group_id}.csv", index=False)
		np.save(f"{loss_dir}/{encoder}_{group_id}.npy", np.array(list_loss))

	if torch.cuda.is_available():
		torch.cuda.empty_cache()

	return {
		"best_epoch": best_epoch if use_val else None,
		"best_val_score": best_score if use_val else None,
		"stopped_epoch": stopped_epoch,
		"epochs_run": len(list_loss),
		"loss": list_loss,
		"val_history": history,
	}

In [16]:
# [MEJORA 1] Hiperparámetros de entrenamiento y partición de pares en un solo lugar
SEED = 42
WEIGHT_DECAY = weight_decay         # antes fijo dentro de la función
HEADS = 4                    # solo GAT / GATE

# Selección de modelo (usa anclajes -> supervisión; poner False para el protocolo no supervisado)
USE_VALIDATION = False
VAL_FRAC = 0.2
EVAL_EVERY = 5
SELECT_METRIC = "mrr"        # "mrr" | "hit@1" | "hit@5" | "hit@10" | "hit@50"
RETRIEVAL = "l2"             # "l2" (como el paper y la evaluación final) | "cosine"

# Objetivo de entrenamiento (valores por defecto = comportamiento previo / paper)
DECODER = "dot"              # "dot" | "cosine" (con TAU y sesgo aprendible)
TAU = 0.2
NEG_RATIO = 1.0              # negativos por positivo
AUG_P = 0.0                  # probabilidad de eliminar cada arista al codificar (experimento A/B)
GRAD_CLIP = None             # p. ej. 1.0

# Con validación el criterio se mide en épocas: 50 = 10 evaluaciones sin mejora con EVAL_EVERY = 5
PATIENCE = 50 if USE_VALIDATION else patience
WARMUP = 10

# Nombre en `encoders` -> tipo de convolución de model_PyG.TGAE
ENCODER_TO_CONV = {"GCNRes": "gcnres", "GIN": "gin", "GINE": "gine",
				   "GCN": "gcn", "GAT": "gatv2", "GATE": "gatv2e"}
# Evaluation
KS = (1, 5, 10, 50)

all_pairs = df_node_alignment_truth.values.astype(np.int64)   # [K, 2]: (nodo en grafo 1, nodo en grafo 2)
if USE_VALIDATION:
	VAL_PAIRS, EVAL_PAIRS = make_val_split(all_pairs, val_frac=VAL_FRAC, seed=SEED)
else:
	VAL_PAIRS, EVAL_PAIRS = None, all_pairs
df_eval_truth = pd.DataFrame(EVAL_PAIRS)
print("Pares totales:", len(all_pairs), "| validación:", 0 if VAL_PAIRS is None else len(VAL_PAIRS),
	  "| evaluación:", len(EVAL_PAIRS))

Pares totales: 1118 | validación: 0 | evaluación: 1118


In [17]:
encoders

['GCNRes', 'GIN', 'GINE', 'GCN', 'GAT', 'GATE']

In [18]:
list_train_set

[['douban_1', 'douban_2']]

In [19]:
list_train_loader

[{'douban_1': Data(x=[3906, 538], edge_index=[2, 16328], edge_attr=[16328, 2]),
  'douban_2': Data(x=[1118, 538], edge_index=[2, 3022], edge_attr=[3022, 2])}]

In [20]:
num_exp = 10 # Change

list_exp = []

for n_exp in range(num_exp):
	print(n_exp)
	#---
	# Train
	#---

	# list_runtime = []
	# list_fit_info = []
	for encoder in encoders:
		for i, train_loader in enumerate(list_train_loader):
			group_id = groups_id[i]
			subgroups_id_ = subgroups_id[group_id]
			train_set = list_train_set[i]

			no_samples = len(train_set)
			input_dim = train_loader[train_set[0]].num_node_features
			edge_dim = train_loader[train_set[0]].edge_attr.size(1) if encoder in ("GINE", "GATE") else None

			# [MEJORA 4a] Semilla ANTES de construir el modelo: antes cada encoder continuaba el flujo aleatorio
			# del anterior (resultados dependientes del orden de `encoders`) y los pesos iniciales no se repetían.
			set_seed(SEED + n_exp)
			model = TGAE(ENCODER_TO_CONV[encoder], NUM_HIDDEN_LAYERS, input_dim, HIDDEN_DIM,
						output_feature_size, edge_dim=edge_dim, heads=HEADS).to(device)

			print("Fitting model")
			print(encoder, dataset, train_set, lr, epochs, input_dim, output_feature_size, no_samples)

			# [MEJORA 4b] CUDA es asíncrono: sincronizar antes y después para medir el tiempo real.
			# Nota: con validación activa, el tiempo incluye las evaluaciones periódicas.
			if device.type == "cuda":
				torch.cuda.synchronize(device)
			start_time = time.perf_counter()

			info_fit = fit_TGAE_subgraph(
				model, encoder, dataset, group_id, subgroups_id_, train_loader, train_set, no_samples,
				epochs, lr, WEIGHT_DECAY,
				val_pairs=VAL_PAIRS, eval_every=EVAL_EVERY, select_metric=SELECT_METRIC, retrieval=RETRIEVAL,
				decoder=DECODER, tau=TAU, neg_ratio=NEG_RATIO, aug_p=AUG_P, grad_clip=GRAD_CLIP,
				patience=PATIENCE, warmup=WARMUP,
			)

			if device.type == "cuda":
				torch.cuda.synchronize(device)
			end_time = time.perf_counter()
			execution_time = end_time - start_time
			list_exp.append([raw_data_file, encoder, output_feature_size, NUM_HIDDEN_LAYERS, n_exp + 1, execution_time])
			
			# list_runtime.append(execution_time)
			# list_fit_info.append({"encoder": encoder, "group_id": group_id,
			# 					**{k: v for k, v in info_fit.items() if k not in ("loss", "val_history")}})

			# [MEJORA 4c] Liberar el modelo (12 capas x 1024) antes del siguiente encoder
			del model
			if device.type == "cuda":
				torch.cuda.empty_cache()
	#---
	# Similarity analysis
	#---
	k = 1 # Change
	for encoder in encoders:
		for group_id in groups_id:
			df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
			# Id, 0, 1,	2, ..., subgroup_id

			subgroups_id_ = subgroups_id[group_id]

			# Calculate distance matrix (KNN)
			knn = NearestNeighbors(n_neighbors=k, metric="euclidean")

			df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroups_id_[0]]
			x = df_node_embeddings.iloc[:, 1:-1].values

			df_node_alignment = pd.DataFrame()
			df_node_alignment[f"{group_id}_{subgroups_id_[0]}"] = df_node_embeddings["Id"].values

			for subgroup_id in subgroups_id_[1:]:
				df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroup_id]
				y = df_node_embeddings.iloc[:, 1:-1].values

				knn.fit(y)
				distances, indices = knn.kneighbors(x)
				indices = indices.squeeze() # (N,)

				df_node_alignment[f"{group_id}_{subgroup_id}"] = df_node_embeddings["Id"].values[indices]
				
				""" df_node_alignment[f"distances"] = distances
				avg_distances = df_node_alignment["distances"].mean()
				df_node_alignment = df_node_alignment[df_node_alignment["distances"] <= avg_distances].iloc[:, :-1]
				df_node_alignment """
			
			# Save previus node alignment
			df_node_alignment.to_csv(f"experiments/output/{exp}/filter_raw/node_alignment_{encoder}_{group_id}.csv")
			# display(df_node_alignment)

			# Find node alignment for all datasets
			""" df_node_alignment_filter = df_node_alignment[df_node_alignment.nunique(axis=1) == 1]
			list_count_meta.append([encoder, group_id, len(df_node_alignment_filter), len(df_node_alignment)])
			# print(len(df_node_alignment_filter))
			# display(df_node_alignment_filter)
			print(f"{encoder}-{group_id}: {len(df_node_alignment_filter)}")
			
			# Save common node id
			# common_node_id = df_node_alignment_filter.iloc[:, 0].values
			# np.save(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy", np.array(common_node_id)) """

			# Save common node id as .csv
			df_common_node_id = df_node_alignment.iloc[:, [0, 1]].copy() # df_common_node_id = df_node_alignment_filter.iloc[:, [0, 1]].copy()
			df_common_node_id.sort_values(by=df_common_node_id.columns[0], inplace=True)
			df_common_node_id.to_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", index=False, header=False)

	# Intersection and Union
	""" list_common_node_id = []
	for encoder in encoders:
		for group_id in groups_id:
			# Read common node
			common_node_id = np.load(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.npy")
			list_common_node_id.append(common_node_id)

		common_node_id_union = list(set.union(*map(set, list_common_node_id)))
		common_node_id_intersection = list(set.intersection(*map(set, list_common_node_id)))

		# Save common node id
		np.save(f"experiments/output/{exp}/common_nodes/{encoder}_union.npy", np.array(common_node_id_union))
		np.save(f"experiments/output/{exp}/common_nodes/{encoder}_intersection.npy", np.array(common_node_id_intersection)) """

	#---
	# Evaluation
	#---
	# list_accuracy = []
	# list_encoder = []

	for j, encoder in enumerate(encoders):
		for group_id in groups_id:
			# Read node alignment
			df_common_node_id = pd.read_csv(f"experiments/output/{exp}/common_nodes/{encoder}_{group_id}.csv", header=None)
			# df_common_node_id[[0, 1]] = np.sort(df_common_node_id[[0, 1]], axis=1)

			# Evaluation
			df_matching = df_common_node_id.merge(df_node_alignment_truth, on=[0, 1], how="inner")

			# Accuracy
			accuracy = len(df_matching) / len(df_node_alignment_truth)
			
			list_exp[len(encoders) * n_exp + j].append(accuracy)
			# list_accuracy.append(accuracy)
			# list_encoder.append(encoder)

	#---
	# Evaluation (based on PlanetAlign)
	#---

	# Get node embeddings
	dict_node_embeddings = {}
	for encoder in encoders:
		list_node_embeddings = []
		for group_id in groups_id:
			df_node_embeddings_concat = pd.read_csv(f"experiments/output/{exp}/node_embeddings/{encoder}_{group_id}.csv", dtype={"subgroup_id": "string"})
			# Id, 0, 1,	2, ..., subgroup_id

			for subgroup_id in subgroups_id[group_id]:
				df_node_embeddings = df_node_embeddings_concat[df_node_embeddings_concat["subgroup_id"] == subgroup_id]
				list_node_embeddings.append(df_node_embeddings)
		dict_node_embeddings[encoder] = list_node_embeddings

	# Evaluation
	# list_metrics = []

	df_node_alignment_truth_ = torch.tensor(df_node_alignment_truth.values)

	for j, encoder in enumerate(encoders):
		list_node_embeddings = dict_node_embeddings[encoder]
		# Id, 0, 1,	2, ..., subgroup_id

		emb1 = list_node_embeddings[0].iloc[:, 1:-1]
		emb2 = list_node_embeddings[1].iloc[:, 1:-1]

		emb1_ = torch.tensor(emb1.values)
		emb2_ = torch.tensor(emb2.values)

		# Similarity matrix
		# Option 1
		S = pairwise_cosine_similarity(emb1_, emb2_)

		# Option 2
		""" S = cosine_similarity(emb1, emb2)
		S = torch.tensor(S) """

		# Option 3
		""" S = pairwise_distances(emb1, emb2, metric="euclidean")
		S = torch.tensor(S)
		S = -S # convert distances matrix to similarity matrix """
		
		# Option 4
		""" S = torch.cdist(emb1_, emb2_, 2)
		S = -S # convert distances matrix to similarity matrix """

		hits = hits_ks_scores(S, df_node_alignment_truth_, ks=KS, mode="mean")
		mrr = mrr_score(S, df_node_alignment_truth_, mode="mean")
		print(encoder, hits, mrr)

		list_exp[len(encoders) * n_exp + j] += list(hits.values()) + [mrr]

0
Fitting model
GCNRes exp108 ['douban_1', 'douban_2'] 0.0001 500 538 512 2


Epoch: 165:  33%|███▎      | 164/500 [00:16<00:34,  9.84it/s, Loss: 0.4460]


Early stopping (pérdida) en la época 165
Fitting model
GIN exp108 ['douban_1', 'douban_2'] 0.0001 500 538 512 2


Epoch: 99:  20%|█▉        | 98/500 [00:25<01:44,  3.84it/s, Loss: 0.4071]


Early stopping (pérdida) en la época 99
Fitting model
GINE exp108 ['douban_1', 'douban_2'] 0.0001 500 538 512 2


...

Early stopping (pérdida) en la época 49
Fitting model
GAT exp108 ['douban_1', 'douban_2'] 0.0001 500 538 512 2


Epoch: 131:  26%|██▌       | 130/500 [00:24<01:08,  5.41it/s, Loss: 0.4006]


Early stopping (pérdida) en la época 131
Fitting model
GATE exp108 ['douban_1', 'douban_2'] 0.0001 500 538 512 2


Epoch: 131:  26%|██▌       | 130/500 [00:25<01:12,  5.09it/s, Loss: 0.3990]


Early stopping (pérdida) en la época 131
GCNRes {1: 0.27191413193941116, 5: 0.4660107344388962, 10: 0.5559034049510956, 50: 0.7616278827190399} 0.3691936731338501
GIN {1: 0.34123434126377106, 5: 0.5138640403747559, 10: 0.5992844253778458, 50: 0.7830948233604431} 0.42854610085487366
GINE {1: 0.23792486637830734, 5: 0.3783542066812515, 10: 0.45796066522598267, 50: 0.6614490151405334} 0.31228435784578323
GCN {1: 0.25402503460645676, 5: 0.4440965950489044, 10: 0.5326475948095322, 50: 0.7428443431854248} 0.3486499786376953
GAT {1: 0.25223612785339355, 5: 0.440518781542778, 10: 0.5317531377077103, 50: 0.7307692468166351} 0.3462388962507248
GATE {1: 0.2616279050707817, 5: 0.44901609420776367, 10: 0.5344364941120148, 50: 0.7410554587841034} 0.3526886850595474


### Summary

In [21]:
list_exp

[['douban',
  'GCNRes',
  512,
  12,
  1,
  17.835656721959822,
  0.2808586762075134,
  0.26654740422964096,
  0.4539355933666229,
  0.5402504354715347,
  0.7406082451343536,
  0.3598587214946747],
 ['douban',
  'GIN',
  512,
  12,
  1,
  26.788133049965836,
  0.42486583184257604,
  0.3443649411201477,
  0.5156529396772385,
  0.5970482975244522,
  0.7902504205703735,
  0.43031586706638336],
 ['douban',
  'GINE',
  512,
  12,
  1,
  37.25843173393514,
  0.28622540250447226,
  0.22719141095876694,
  0.3649373948574066,
  0.447674423456192,
  0.6444543749094009,
  0.3002644404768944],
 ['douban',
  'GCN',
  512,
  12,
  1,
  7.7384405840421095,
  0.2182468694096601,
  0.2580500841140747,
  0.4414132535457611,
  0.5263863950967789,
  0.737030416727066,
  0.3491926044225693],
 ['douban',
  'GAT',
  512,
  12,
  1,
  14.210501609952189,
  0.2817531305903399,
  0.273255817592144,
  0.44991055130958557,
  0.5348837077617645,
  0.743291586637497,
  0.3622090443968773],
 ['douban',
  'GATE',
  5

In [22]:
columns = ["Dataset", "Encoder", "Dimension", "Layer", "Seed", "Runtime", "Accuracy"] + [f"Hits@{k}" for k in list(hits.keys())] +  ["MRR"]
df_summary = pd.DataFrame(list_exp, columns=columns)
df_summary.sort_values(by=["Encoder", "Seed"], inplace=True)
df_summary

,Dataset,Encoder,Dimension,Layer,Seed,Runtime,Accuracy,Hits@1,Hits@5,Hits@10,Hits@50,MRR
4,douban,GAT,512,12,1,14.210502,0.281753,0.273256,0.449911,0.534884,0.743292,0.362209
10,douban,GAT,512,12,2,22.081436,0.287120,0.253578,0.438283,0.535778,0.748211,0.347881
16,douban,GAT,512,12,3,21.201697,0.309481,0.265206,0.436494,0.529964,0.743292,0.354465
22,douban,GAT,512,12,4,17.556015,0.296959,0.282648,0.454383,0.538462,0.745975,0.370642
28,douban,GAT,512,12,5,20.537852,0.296959,0.254025,0.444991,0.528623,0.740608,0.347251
34,douban,GAT,512,12,6,21.539454,0.292487,0.264758,0.447227,0.527728,0.745528,0.356227
40,douban,GAT,512,12,7,26.089180,0.298748,0.265206,0.443202,0.531753,0.747764,0.356115
46,douban,GAT,512,12,8,18.472737,0.285331,0.255367,0.434705,0.511628,0.737030,0.344600
52,douban,GAT,512,12,9,15.024098,0.295170,0.282648,0.459750,0.542487,0.757603,0.372800
58,douban,GAT,512,12,10,25.249483,0.298748,0.252236,0.440519,0.531753,0.730769,0.346239


In [23]:
df_text = df_summary.map(lambda x: f"{x}".replace(".", ","))
df_text

,Dataset,Encoder,Dimension,Layer,Seed,Runtime,Accuracy,Hits@1,Hits@5,Hits@10,Hits@50,MRR
4,douban,GAT,512,12,1,"14,210501609952189","0,2817531305903399","0,273255817592144","0,44991055130958557","0,5348837077617645","0,743291586637497","0,3622090443968773"
10,douban,GAT,512,12,2,"22,08143561705947","0,28711985688729874","0,253577820956707","0,43828265368938446","0,5357781648635864","0,7482110857963562","0,34788139164447784"
16,douban,GAT,512,12,3,"21,201697165030055","0,3094812164579606","0,2652057185769081","0,43649373948574066","0,5299642235040665","0,743291586637497","0,35446450859308243"
22,douban,GAT,512,12,4,"17,55601530300919","0,29695885509839","0,28264757990837097","0,4543828219175339","0,5384615510702133","0,7459749579429626","0,3706424832344055"
28,douban,GAT,512,12,5,"20,537851719069295","0,29695885509839","0,25402504950761795","0,4449910670518875","0,5286225527524948","0,7406082153320312","0,34725096076726913"
34,douban,GAT,512,12,6,"21,53945426410064","0,2924865831842576","0,26475849747657776","0,4472271800041199","0,5277280956506729","0,7455277144908905","0,35622725635766983"
40,douban,GAT,512,12,7,"26,089180122013204","0,29874776386404295","0,2652057185769081","0,4432021379470825","0,5317531228065491","0,7477638721466064","0,3561154678463936"
46,douban,GAT,512,12,8,"18,472737182979472","0,2853309481216458","0,2553667277097702","0,43470484018325806","0,5116278976202011","0,737030416727066","0,3445996344089508"
52,douban,GAT,512,12,9,"15,024098486057483","0,295169946332737","0,28264757990837097","0,4597495496273041","0,5424865782260895","0,757602870464325","0,372799851000309"
58,douban,GAT,512,12,10,"25,249482684070244","0,29874776386404295","0,25223612785339355","0,440518781542778","0,5317531377077103","0,7307692468166351","0,3462388962507248"


In [24]:
print(NUM_HIDDEN_LAYERS)
print(HIDDEN_DIM)
print(output_feature_size)

12
[1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024, 1024]
512
